# Route Visualization from Test.txt

This notebook parses routing decisions from Test.txt and creates two visualizations:
1. Actual route taken by the vehicle
2. Planned route from a specific subproblem

In [ ]:
import re
import json
import matplotlib.pyplot as plt

THESIS_COLORS = {
    "Pale Fern": "#D9E5D6",
    "Sage Green": "#84A579",
    "Dark Spruce": "#344E41",
    "Warm Red": "#D1495B",
    "Ochre Gold": "#EDAE49",
    "Blue Accent": "#526ECA",
    "Mist Grey": "#E6E6E6",
}

In [ ]:
def parse_test_file(filepath):
    """
    Parse Test.txt file to extract both actual route and planned decisions.
    Returns: list of routing blocks, each containing actual actions and planned route.
    """
    with open(filepath, 'r') as f:
        content = f.read()
    
    blocks = content.split('=== ROUTING ANALYSIS')
    routing_data = []
    
    for block in blocks[1:]:  # Skip first empty split
        if not block.strip():
            continue
        
        # Extract current station
        station_match = re.search(r'at station S(\d+)', block)
        if not station_match:
            continue
        
        current_station = station_match.group(1)
        
        # Extract actual actions (Load/Unload/Maintenance)
        actions = []
        load_match = re.search(r'Load:\s+(\d+\.?\d*)', block)
        unload_match = re.search(r'Unload:\s+(\d+\.?\d*)', block)
        maint_match = re.search(r'Maintenance:\s+(\d+\.?\d*)\s+minutes', block)
        
        if load_match and float(load_match.group(1)) > 0:
            actions.append(f"Load: {load_match.group(1)}")
        if unload_match and float(unload_match.group(1)) > 0:
            actions.append(f"Unload: {unload_match.group(1)}")
        if maint_match and float(maint_match.group(1)) > 0:
            actions.append(f"Maint: {maint_match.group(1)}m")
        
        # Extract planned route from decisions (x[i,j,v,t] variables)
        planned_route = []
        x_pattern = r'x\[(-?\d+),(-?\d+),0,(\d+)\]\s*=\s*1\.00'
        x_matches = re.findall(x_pattern, block)
        
        for from_station, to_station, period in x_matches:
            planned_route.append({
                'from': from_station,
                'to': to_station,
                'period': period
            })
        
        # Sort by period
        planned_route.sort(key=lambda x: int(x['period']))
        
        routing_data.append({
            'current_station': current_station,
            'actions': actions,
            'planned_route': planned_route
        })
    
    return routing_data

print("parse_test_file() function defined")

In [ ]:
def load_station_coordinates(json_filepath):
    """
    Load station coordinates from JSON (keeps lat/lon format).
    Returns: dict mapping station_id to {lat, lon} coordinates
    """
    with open(json_filepath, 'r') as f:
        data = json.load(f)
    
    station_coords = {}
    
    for station in data['stations']:
        station_id = str(station['id'])
        lat = station['location'][0]
        lon = station['location'][1]
        station_coords[station_id] = {'lat': lat, 'lon': lon}
    
    return station_coords

print("load_station_coordinates() function defined")

In [ ]:
def visualize_actual_route(routing_data, station_coords):
    """
    Visualize the actual route taken (sequence of stations visited).
    """
    all_lats = [coords['lat'] for coords in station_coords.values()]
    all_lons = [coords['lon'] for coords in station_coords.values()]
    
    lat_min, lat_max = min(all_lats), max(all_lats)
    lon_min, lon_max = min(all_lons), max(all_lons)
    lat_padding = (lat_max - lat_min) * 0.1
    lon_padding = (lon_max - lon_min) * 0.1
    
    fig, ax = plt.subplots(figsize=(16, 12))
    
    ax.set_title('Actual Route Taken', fontsize=18, color=THESIS_COLORS["Dark Spruce"], fontweight='bold')
    ax.set_xlim(lon_min - lon_padding, lon_max + lon_padding)
    ax.set_ylim(lat_min - lat_padding, lat_max + lat_padding)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--', color='gray')
    ax.set_facecolor('#f8f9fa')
    
    lon_offset = 0.003  # Offset for action text
    
    # Plot the sequence of stations visited
    for i, block in enumerate(routing_data):
        current_id = block['current_station']
        
        if current_id not in station_coords:
            continue
        
        lon = station_coords[current_id]['lon']
        lat = station_coords[current_id]['lat']
        
        # Plot station marker
        ax.text(lon, lat, f"S{current_id}", color="white", weight='bold', fontsize=11,
                ha='center', va='center',
                bbox={'facecolor': THESIS_COLORS["Blue Accent"], 
                      'edgecolor': THESIS_COLORS["Dark Spruce"], 
                      'boxstyle':'round,pad=0.4'})
        
        # Plot actions
        if block['actions']:
            action_text = "\n".join(block['actions'])
            ax.text(lon + lon_offset, lat, action_text, fontsize=9,
                    verticalalignment='center',
                    bbox={'facecolor': THESIS_COLORS["Mist Grey"], 
                          'edgecolor': THESIS_COLORS["Sage Green"], 
                          'boxstyle': 'round,pad=0.5',
                          'alpha': 0.95})
        
        # Draw arrow to next station
        if i < len(routing_data) - 1:
            next_id = routing_data[i + 1]['current_station']
            if next_id in station_coords and current_id != next_id:
                next_lon = station_coords[next_id]['lon']
                next_lat = station_coords[next_id]['lat']
                
                ax.annotate("", xy=(next_lon, next_lat), xytext=(lon, lat),
                            arrowprops=dict(arrowstyle="->", lw=3,
                                            color=THESIS_COLORS["Ochre Gold"]))
                
                # Add sequence number
                mid_lon = (lon + next_lon) / 2
                mid_lat = (lat + next_lat) / 2
                ax.text(mid_lon, mid_lat, str(i+1), color=THESIS_COLORS["Warm Red"], 
                        weight='bold', fontsize=11,
                        bbox={'facecolor': 'white', 'edgecolor': 'none', 'alpha': 0.8, 'pad': 2})
    
    plt.tight_layout()
    plt.show()

print("visualize_actual_route() function defined")

In [ ]:
def visualize_planned_route(routing_block, station_coords):
    """
    Visualize the planned route from a single subproblem's decisions.
    """
    all_lats = [coords['lat'] for coords in station_coords.values()]
    all_lons = [coords['lon'] for coords in station_coords.values()]
    
    lat_min, lat_max = min(all_lats), max(all_lats)
    lon_min, lon_max = min(all_lons), max(all_lons)
    lat_padding = (lat_max - lat_min) * 0.1
    lon_padding = (lon_max - lon_min) * 0.1
    
    fig, ax = plt.subplots(figsize=(16, 12))
    
    current_station = routing_block['current_station']
    ax.set_title(f'Planned Route from Subproblem (at S{current_station})', 
                 fontsize=18, color=THESIS_COLORS["Dark Spruce"], fontweight='bold')
    ax.set_xlim(lon_min - lon_padding, lon_max + lon_padding)
    ax.set_ylim(lat_min - lat_padding, lat_max + lat_padding)
    ax.set_xlabel('Longitude', fontsize=12)
    ax.set_ylabel('Latitude', fontsize=12)
    ax.grid(True, alpha=0.3, linestyle='--', color='gray')
    ax.set_facecolor('#f8f9fa')
    
    # Extract unique stations from planned route
    stations_in_route = set()
    for move in routing_block['planned_route']:
        from_id = move['from']
        to_id = move['to']
        # Skip source (-1) and sink (-2)
        if from_id != '-1' and from_id != '-2':
            stations_in_route.add(from_id)
        if to_id != '-1' and to_id != '-2':
            stations_in_route.add(to_id)
    
    # Plot all stations involved in the route
    for station_id in stations_in_route:
        if station_id in station_coords:
            lon = station_coords[station_id]['lon']
            lat = station_coords[station_id]['lat']
            
            # Highlight current station differently
            if station_id == current_station:
                color = THESIS_COLORS["Warm Red"]
                label = f"S{station_id}*"
            else:
                color = THESIS_COLORS["Blue Accent"]
                label = f"S{station_id}"
            
            ax.text(lon, lat, label, color="white", weight='bold', fontsize=11,
                    ha='center', va='center',
                    bbox={'facecolor': color, 
                          'edgecolor': THESIS_COLORS["Dark Spruce"], 
                          'boxstyle':'round,pad=0.4'})
    
    # Plot the planned movements
    for i, move in enumerate(routing_block['planned_route']):
        from_id = move['from']
        to_id = move['to']
        period = move['period']
        
        # Skip movements involving source/sink or staying at same station
        if from_id in ['-1', '-2'] or to_id in ['-1', '-2']:
            continue
        if from_id == to_id:
            continue
        
        if from_id in station_coords and to_id in station_coords:
            from_lon = station_coords[from_id]['lon']
            from_lat = station_coords[from_id]['lat']
            to_lon = station_coords[to_id]['lon']
            to_lat = station_coords[to_id]['lat']
            
            # Draw arrow
            ax.annotate("", xy=(to_lon, to_lat), xytext=(from_lon, from_lat),
                        arrowprops=dict(arrowstyle="->", lw=2.5,
                                        color=THESIS_COLORS["Sage Green"],
                                        alpha=0.7))
            
            # Add period label
            mid_lon = (from_lon + to_lon) / 2
            mid_lat = (from_lat + to_lat) / 2
            ax.text(mid_lon, mid_lat, f"t={period}", color=THESIS_COLORS["Dark Spruce"], 
                    weight='bold', fontsize=10,
                    bbox={'facecolor': 'white', 'edgecolor': THESIS_COLORS["Sage Green"], 
                          'alpha': 0.9, 'pad': 2})
    
    plt.tight_layout()
    plt.show()

print("visualize_planned_route() function defined")

## Parse Test.txt and Load Station Data

In [ ]:
# File paths
test_file = r'C:\Users\ingvivsu\OneDrive - NTNU\Master\FOMOsim\policies\sjovik_sund\output\Test.txt'
json_file = r'C:\Users\ingvivsu\OneDrive - NTNU\Master\FOMOsim\policies\sjovik_sund\output\instance_data\TD_W34_67.json'

# Parse the file
routing_data = parse_test_file(test_file)
print(f"Parsed {len(routing_data)} routing blocks")

# Load station coordinates
station_coords = load_station_coordinates(json_file)
print(f"Loaded {len(station_coords)} station coordinates")

## 1. Visualize Actual Route Taken

In [ ]:
visualize_actual_route(routing_data, station_coords)

## 2. Visualize Planned Route from First Subproblem

In [ ]:
# Visualize the first subproblem's planned route
# Change index to see different subproblems (0, 1, 2, etc.)
subproblem_index = 0
visualize_planned_route(routing_data[subproblem_index], station_coords)

## Browse Different Subproblems

You can visualize any subproblem by changing the index:

In [ ]:
# Show what stations are analyzed in each subproblem
for i, block in enumerate(routing_data):
    print(f"Block {i}: Station S{block['current_station']} - Actions: {', '.join(block['actions']) if block['actions'] else 'None'}")

In [ ]:
# Visualize a different subproblem (change the index)
subproblem_index = 5
visualize_planned_route(routing_data[subproblem_index], station_coords)